In [1]:
import bw2data, bw2io, bw2calc
from bw_timex import TimexLCA
from bw_temporalis import TemporalDistribution, easy_timedelta_distribution
import numpy as np
from datetime import datetime
import os
import re
import pandas as pd
import numpy as np
import pickle

In [2]:
import sys
sys.path.append('../utils/') 
from elec_builder import *

In [3]:
# activate the bw project
bw2data.projects.set_current("ei311")
#for db in bw2data.databases:
#    print(db, len(bw2data.Database(db)))

### 1. building PV_foreground 
- PV: electricity production, photovoltaic, commercial:::  CN|US

In [ ]:
pv_loc = ['CN', 'US']

xx = build_dynamic_electricity_all(
    locations = pv_loc , 
    pathways = [ "SSP1-VLLO" , "SSP2-M", "SSP5-H" ],
    years = [2030, 2040, 2050], 
    elec_act = 'electricity production, photovoltaic, commercial',
    ref_name = 'market for electricity, PV, low voltage',
    fg_db_name="elec_PV_foreground",
    flush_fg_db = True
)

# not tested: assign_td_from_results_dict(results_dict = xx, elec_td_year=10)

In [5]:
pv_db = bw2data.Database("elec_PV_foreground")
len(pv_db)

18

In [6]:
act_list = list(pv_db)[2:4]
for act in act_list:
    tech_excs = list(act.technosphere())
    print(len(tech_excs))
    exc = tech_excs[0]
    print(exc)

1
Exchange: 1 kilowatt hour 'electricity production, photovoltaic, commercial' (kilowatt hour, CN, None) to 'market for electricity, PV, low voltage, CN, SSP5-H, 2030' (kWh, CN, None)>
1
Exchange: 1 kilowatt hour 'electricity production, photovoltaic, commercial' (kilowatt hour, US, None) to 'market for electricity, PV, low voltage, US, SSP2-M, 2030' (kWh, US, None)>


In [7]:
rows = []   # collect results for all acts
act_list = list(pv_db)   
for act in act_list: 
    print(act)

    name = act.get("name")
    name_parts = [p.strip() for p in name.split(",")]
    # run static LCI + premise_GWP vs. pGWP100 first: 
    pgwp_fixedco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - fixed-AGWPCO2")
    pgwp_dpco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - dp-AGWPCO2")
    gwp =  ('ecoinvent-3.11', 'IPCC 2021 (incl. biogenic CO2)', 'climate change: total (incl. biogenic CO2, incl. SLCFs)', 'global warming potential (GWP100)')   

    lca_gwp = bw2calc.lca.LCA({act: 1}, method=gwp)
    lca_gwp.lci(); lca_gwp.lcia()
    score_gwp = float(lca_gwp.score)

    lca_fixed = bw2calc.lca.LCA({act: 1}, method=pgwp_fixedco2)
    lca_fixed.lci(); lca_fixed.lcia()
    score_fixedco2 = float(lca_fixed.score)

    lca_dp = bw2calc.lca.LCA({act: 1}, method=pgwp_dpco2)
    lca_dp.lci(); lca_dp.lcia()
    score_dpco2 = float(lca_dp.score)

    print(score_gwp, score_fixedco2, score_dpco2) 

    # ---- store results for this activity ----
    rows.append({
        "Activity": name,                      # index value later
        "gwp100": score_gwp,
        "pGWP100_fixedCO2": score_fixedco2,
        "pGWP100_dpCO2": score_dpco2,
    })

'market for electricity, PV, low voltage, CN, SSP2-M, 2050' (kWh, CN, None)
0.020243585508028274 0.02066663487021566 0.01964426955749808
'market for electricity, PV, low voltage, US, SSP1-VLLO, 2040' (kWh, US, None)
0.009084464179143804 0.00867927162369792 0.008735305689131445
'market for electricity, PV, low voltage, CN, SSP1-VLLO, 2030' (kWh, CN, None)
0.032006134483650896 0.03188413067097 0.031263990161578284
'market for electricity, PV, low voltage, CN, SSP1-VLLO, 2040' (kWh, CN, None)
0.013980313940796898 0.01332608521172227 0.013412038687364831
'market for electricity, PV, low voltage, CN, SSP5-H, 2040' (kWh, CN, None)
0.029716793273824475 0.03756581378248621 0.02855040369815805
'market for electricity, PV, low voltage, US, SSP5-H, 2040' (kWh, US, None)
0.01882845669677395 0.02381945490087417 0.018102949344957764
'market for electricity, PV, low voltage, US, SSP2-M, 2030' (kWh, US, None)
0.02074043361965896 0.023453046660848766 0.020169279390328317
'market for electricity, PV, lo

In [8]:
import pandas as pd

df_scores = pd.DataFrame(rows)
df_scores = df_scores.set_index("Activity")
df_scores

,gwp100,pGWP100_fixedCO2,pGWP100_dpCO2
Activity,,,
"market for electricity, PV, low voltage, CN, SSP2-M, 2050",0.020244,0.020667,0.019644
"market for electricity, PV, low voltage, US, SSP1-VLLO, 2040",0.009084,0.008679,0.008735
"market for electricity, PV, low voltage, CN, SSP1-VLLO, 2030",0.032006,0.031884,0.031264
"market for electricity, PV, low voltage, CN, SSP1-VLLO, 2040",0.013980,0.013326,0.013412
"market for electricity, PV, low voltage, CN, SSP5-H, 2040",0.029717,0.037566,0.028550
"market for electricity, PV, low voltage, US, SSP5-H, 2040",0.018828,0.023819,0.018103
"market for electricity, PV, low voltage, US, SSP2-M, 2030",0.020740,0.023453,0.020169
"market for electricity, PV, low voltage, US, SSP1-VLLO, 2030",0.020106,0.020041,0.019651
"market for electricity, PV, low voltage, CN, SSP2-M, 2030",0.032915,0.037194,0.031987


In [9]:
df_scores.sort_values(by=['Activity']).to_excel("dp-LCI_output/staticLCI_(p)GWP100/PV_CN_US_staticLCI_threeGWP100.xlsx")

### 2. building dpLCI

In [4]:
pv_db = bw2data.Database('elec_PV_foreground')
len(list(pv_db) )

18

In [5]:
database_dates = {
    'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP5-H_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2050 2025-11-22': datetime.strptime("2050", "%Y"),
    
    "elec_PV_foreground": "dynamic", # flag databases that should be temporally distributed with "dynamic"
}


#### run seperately for each MY if memory/storage error

In [6]:
pv_db_203050 = [
    act for act in pv_db 
    if "2050" in str(act.get('name', '')) or  "2030" in str(act.get('name', ''))
]
len(list(pv_db_203050))

12

In [7]:
dp_results = {}
for act in list(pv_db_203050): 
    print(act)
    #### dynamic LCI:: 
    assign_td_from_foreground_db( 
            select_act = act,
            elec_td_year=10,
            resolution="Y",
            kind="uniform",
            fg_db_name="elec_PV_foreground",
            verbose = True
         )
    
    tlca = run_dp_timex_lca(foreground_act = act ,   
            pathway = None,
            year = None,
            method = None,
            database_dates = database_dates, #None not working, has to incl. all 9 background DB ... 
            temporal_grouping="year", 
            method_prefix = "Climate Change prospective GWP100",
            method_suffix = "pGWP100 - fixed-AGWPCO2", 
            fg_db_name = 'elec_PV_foreground' )


    tlca.lci()
    tlca.dynamic_inventory.shape
    # dyn-foreground LCI + static background LCI + pGWP100
    lca_0 = tlca.base_score 
    print(lca_0)
    # dpLCI (dyn-foreground LCI + dyn background LCI) +  pGWP100
    tlca.static_lcia()
    lca_1 = tlca.static_score
    print(lca_1)
    act_name = act.get("name")
    dp_results[act_name] = {
        "dyn FG LCI static BG p-LCI, pGWP100-fixedCO2": float(lca_0),   # static BG LCI + dyn FG LCI
        "full dp-LCI, pGWP100-fixedCO2": float(lca_1),      # dyn BG LCI + dyn FG LCI
    }
    #### now save all dyLCI flows to pandas 
    df = tlca.dynamic_inventory_df
    ##### important to convert flow and act as str to excel 
    df["flow"] = df["flow"].astype(str)
    df["activity"] = df["activity"].astype(str)

    ### export df to dp-LCI_output folder, using the act name as the excel name 
    out_dir = "dp-LCI_output/pv_dpLCI"
    os.makedirs(out_dir, exist_ok=True)    
    raw_name = act["name"]
    safe_name = re.sub(r"[^A-Za-z0-9_\-()]+", "_", raw_name)   # replace spaces/special chars
    
    excel_path = os.path.join(out_dir, f"{safe_name}.xlsx")
    
    df.to_excel(excel_path, index=False)    
    print(f"✔ Exported dynamic inventory DF for '{raw_name}' → {excel_path}")


out_dir = "dp-LCI_output/PV_dpLCI"
os.makedirs(out_dir, exist_ok=True)

pickle_path = os.path.join(out_dir, "PV_dp_results_MY2030_2050.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(dp_results, f)

print(f"✔ Saved dp_results dictionary → {pickle_path}")

2026-03-30 19:56:33.970 | INFO     | bw_timex.timex_lca:__init__:114 - Initializing TimexLCA object...
2026-03-30 19:56:33.971 | INFO     | bw_timex.timex_lca:__init__:136 - Collecting node infos...


'market for electricity, PV, low voltage, CN, SSP2-M, 2030' (kWh, CN, None)
TD applied to 'market for electricity, PV, low voltage, CN, SSP2-M, 2030' → Exchange: 1 kilowatt hour 'electricity production, photovoltaic, commercial' (kilowatt hour, CN, None) to 'market for electricity, PV, low voltage, CN, SSP2-M, 2030' (kWh, CN, None)>
for the activity 'market for electricity, PV, low voltage, CN, SSP2-M, 2030' (kWh, CN, None),it's under SSP-SSP2-M, year-2030 
 we'll use LCIA ('Climate Change prospective GWP100', 'SSP245', 'MY2030', 'pGWP100 - fixed-AGWPCO2')  with background database.datetime = {'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.datetime(2040, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.datetime(2050, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22': datetim

2026-03-30 19:59:07.729 | INFO     | bw_timex.timex_lca:build_timeline:216 - No edge filter function provided. Skipping all edges in background databases.
2026-03-30 20:00:13.364 | INFO     | bw_timex.timex_lca:build_timeline:232 - Calculating base LCA...
2026-03-30 20:00:40.569 | INFO     | bw_timex.timex_lca:build_timeline:242 - Creating activity time mapping...
2026-03-30 20:00:43.261 | INFO     | bw_timex.timeline_builder:__init__:99 - Traversing supply chain graph...


Starting graph traversal


2026-03-30 20:00:45.221 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...


...

Starting graph traversal


2026-03-30 21:53:33.791 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...


Calculation count: 1


2026-03-30 21:53:33.969 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 21:53:33.970 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 21:53:33.970 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 21:53:33.972 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2026-03-30 21:53:33.973 | INFO     | bw_time

0.023453046660848766
0.023967673595498194
✔ Exported dynamic inventory DF for 'market for electricity, PV, low voltage, US, SSP2-M, 2030' → dp-LCI_output/pv_dpLCI/market_for_electricity_PV_low_voltage_US_SSP2-M_2030.xlsx
✔ Saved dp_results dictionary → dp-LCI_output/PV_dpLCI/PV_dp_results_MY2030_2050.pkl
